# Simple Storage Service (S3)

## What you will learn in this course 🧐🧐

By the end of this lecture, you will be able to:

- Explain what S3 is and why it is used as a data lake
- Design a bucket layout using `raw/`, `staging/`, and `curated/` prefixes
- Apply the safety defaults that prevent common S3 mistakes (public access, accidental overwrites, missing encryption)
- Recognise the AWS security, logging, networking, and storage services you will see on the job and in quizzes

<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D01-finpulse-logo.png"
  alt="FinPulse logo"
/>

**FinPulse** is a fintech startup in your training scenario. It tracks several cryptocurrencies and stores each day's open, high, low, close, and volume in a JSON file named `crypto_data.json`. Analysts need that history in one shared place, not on a single laptop.

S3 is AWS object storage. You will use it as FinPulse's **data lake**, with prefixes such as `raw/`, `staging/`, and `curated/` so everyone knows which layer they are reading. This notebook covers how to create a bucket safely and how to lay out those keys before you upload data from Python.

## What Is S3?

S3 (Simple Storage Service) is AWS object storage. Teams use it as the starting point of a data lake because it is cheap, durable, and it can store almost anything, from JSON files to parquet datasets.

<Note type="note">

S3 is not a filesystem. The console shows a folder tree, but S3 stores objects under a single string key like `raw/crypto/crypto_data.json`. The `raw/crypto/` part is a prefix inside that key.

</Note>

### Buckets

Objects in S3 live inside a bucket. Bucket names are globally unique, and you choose a region when you create the bucket.

A typical bucket URL looks like this:

```
https://s3-<region>.amazonaws.com/<bucket-name>
```



## Storage Classes

S3 has multiple storage classes. They trade off price against retrieval speed. For a data lake, raw data usually starts in Standard, then moves to a cheaper class after a fixed number of days.

| Storage class | When to use it | Retrieval time |
|---|---|---|
| S3 Standard | Frequently accessed data | milliseconds |
| S3 Standard-IA | Infrequently accessed, but still needs fast reads | milliseconds |
| S3 One Zone-IA | Like Standard-IA, but stored in a single AZ | milliseconds |
| S3 Intelligent-Tiering | Access patterns are unpredictable | milliseconds |
| S3 Glacier Instant Retrieval | Archive data that still needs fast reads sometimes | milliseconds |
| S3 Glacier Flexible Retrieval | Long-term archive, retrieval in minutes or hours | minutes to hours |
| S3 Glacier Deep Archive | Compliance retention (7–10 years), lowest cost | hours |

Glacier Deep Archive is the cheapest S3 storage class. It is designed for data you almost never access but must keep for years, like regulatory filings or audit logs. A lifecycle rule can move raw data there automatically after a retention threshold.

## Create an S3 Bucket

Go to your AWS console, open **S3**, and follow these steps.


### Step 1 Open the S3 console

1. Sign in to your AWS account at `https://console.aws.amazon.com`
2. In the top right, pick a Region close to you (for example `eu-west-3` for Paris) and keep that Region for every FinPulse bucket and object you create
3. In the search bar, type **S3** and open the service

<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D02-s3-console-landing.png"
  alt="S3 console landing page with Create bucket button"
/>

### Step 2 Enter a name and choose a region

1. Click **Create bucket**
2. Enter a bucket name. Bucket names are globally unique, so add a project prefix to avoid collisions, for example `finpulse-yourname-data`
3. Under **AWS Region**, choose the same Region you picked in Step 1 (for example `eu-west-3`)

<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D02-s3-create-bucket-name-region.png"
  alt="Create bucket form with bucket name and region fields"
/>

<Note type="warning" title="Bucket names are globally unique">

Bucket names must be unique across all AWS accounts worldwide. If the name is already taken, AWS tells you immediately. Adding your name or a short random suffix solves this.

</Note>

### Step 3 Block Public Access (keep it on)

Scroll down to **Block Public Access settings for this bucket**. Leave all four checkboxes enabled.

<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D02-s3-create-bucket-block-access.png"
  alt="Block Public Access settings with all four options enabled"
/>

### Step 4 Create the bucket

Scroll to the bottom and click **Create bucket**. AWS takes a second, then redirects you to the bucket list.

<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D02-s3-bucket-created.png"
  alt="S3 bucket list showing the newly created bucket"
/>

<Note type="tip" title="Region and bucket are linked forever">

You cannot change a bucket’s region after creation. If you create it in the wrong region, you have to delete it and start again. Always confirm the Region selector in the top right of the console before clicking Create bucket.

</Note>

## Structure Your Lake

A bucket is just a container. The useful part is how you organise keys inside it.

A simple data lake layout uses three zones:

- `raw/` for data that lands exactly as it was collected
- `staging/` for cleaned and validated data
- `curated/` for data that is ready for analytic


In [1]:
bucket_name = "your-bucket-name"  # replace with your bucket
key = "raw/crypto/crypto_data.json"
print(f"s3://{bucket_name}/{key}")


s3://your-bucket-name/raw/crypto/crypto_data.json


This is the object key pattern you will use from Python: `raw/crypto/crypto_data.json`.

S3 does not store folders. The console displays `raw/crypto/` as a folder, but the full key is one string: `raw/crypto/crypto_data.json`.


## Keep Data Safe by Default

Permissions decide who can read and write your objects. For FinPulse labs, keep the bucket private by default.

### Block Public Access

Block Public Access is enabled by default on new buckets. Leave it on.

The only common reason to turn it off is when you intentionally host a public static website from S3. That is not what you are doing for FinPulse storage.


To check or update Block Public Access on a bucket you already created:

1. Open the bucket in the S3 console
2. Click the **Permissions** tab
3. Scroll to **Block Public Access (bucket settings)**
4. Click **Edit** if any setting is disabled, re-enable all four options, and save

<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D02-s3-block-public-access.png"
  alt="Bucket Permissions tab with Block Public Access settings all enabled"
/>

<Note type="important">

Block Public Access has been enabled by default on new S3 buckets for years. Keep it enabled.

If you disable it by mistake, you can expose data publicly and create a security incident.

</Note>

## Versioning

Versioning keeps old copies of objects. If you overwrite `raw/crypto/crypto_data.json`, S3 keeps the previous version instead of losing it.

<Note type="tip">

Versioning can increase storage cost because non-current versions are still stored. Pair versioning with a lifecycle rule that expires non-current versions after a fixed number of days.

</Note>


## Lifecycle Rules

Lifecycle rules automate two useful things:

- moving objects to cheaper storage classes after a fixed number of days
- deleting objects or non-current versions after a retention period

Here is what a simple transition rule looks like in Python. In a hands-on stretch task, you will build and apply a rule like this using Boto3.


In [2]:
lifecycle_rule = {
    "ID": "raw-to-ia-30d",
    "Filter": {"Prefix": "raw/"},
    "Status": "Enabled",
    "Transitions": [{"Days": 30, "StorageClass": "STANDARD_IA"}],
}

lifecycle_rule


{'ID': 'raw-to-ia-30d',
 'Filter': {'Prefix': 'raw/'},
 'Status': 'Enabled',
 'Transitions': [{'Days': 30, 'StorageClass': 'STANDARD_IA'}]}

Read this dict as a description of the rule.

Anything under the `raw/` prefix transitions to **Standard-IA** after 30 days.

<Note type="tip">

Give lifecycle rules clear names. `raw-to-ia-30d` is easier to audit than `Rule1` when you revisit the bucket months later.

</Note>


## Encryption

S3 supports encryption at rest and in transit.

- **At rest**: S3 encrypts objects automatically (SSE-S3 is the default on new buckets).
- **In transit**: always use HTTPS when uploading and downloading.
- **SSE-KMS**: use KMS when you need tighter controls and an audit trail for key usage.

<Note type="note">

SSE-KMS can add cost per request. Use it when compliance or audit requirements justify it, not by habit.

</Note>


## Access Control

IAM decides who you are. S3 access control decides what that identity can do on this bucket.

For new workloads, bucket policies are the recommended approach. ACLs still show up in older buckets, so it helps to recognise them.


### Access Control Lists (ACLs)

ACLs are a legacy way to manage access at the object and bucket level.

<Note type="warning">

AWS recommends disabling ACLs for new buckets and using bucket policies instead. The video is included so you can recognise ACL settings on existing accounts.

</Note>


To see the ACL settings for a bucket:

1. Open the bucket in the S3 console
2. Click the **Permissions** tab
3. Scroll to **Access Control List (ACL)**

<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D02-s3-acl-settings.png"
  alt="Bucket Permissions tab showing the ACL section with owner and grantee rows"
/>

For new buckets, the ACL shows only the bucket owner with full control. Leave it that way and use bucket policies for any access grants.

### Bucket Policies

Bucket policies are JSON documents that grant or deny actions on a bucket or on a prefix inside a bucket.

If you want to allow access to `raw/crypto/` but deny access to everything else, a bucket policy is the right tool.


To add or edit a bucket policy:

1. Open the bucket in the S3 console
2. Click the **Permissions** tab
3. Scroll to **Bucket policy** and click **Edit**
4. Paste your policy JSON into the editor and click **Save changes**

<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D02-s3-bucket-policy-editor.png"
  alt="Bucket policy editor with a JSON policy document"
/>

This example policy denies any request that does not use HTTPS. Replace `your-bucket-name` with your actual bucket name.

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Deny",
      "Principal": "*",
      "Action": "s3:*",
      "Resource": [
        "arn:aws:s3:::your-bucket-name",
        "arn:aws:s3:::your-bucket-name/*"
      ],
      "Condition": {
        "Bool": {
          "aws:SecureTransport": "false"
        }
      }
    }
  ]
}
```

## AWS Security and Storage at a Glance


You do not need to configure these services for the FinPulse labs. You do need to recognise their names and what they are for. They appear in Cloud Practitioner exam scenarios and in architecture diagrams on the job.

### Security and governance services

| Service | What it does |
|---|---|
| GuardDuty | Detects suspicious activity in your AWS account |
| Macie | Finds sensitive data in S3 (PII) |
| Inspector | Scans workloads for vulnerabilities |
| Artifact | Compliance reports and agreements |
| Secrets Manager | Stores and rotates secrets |
| AWS Config | Tracks configuration changes and drift |
| CloudTrail | Records AWS API calls (who did what, when, from where) |

### Logging and audit

CloudTrail is account-wide and logs AWS API calls. S3 access logs are bucket-specific and log requests made to a bucket.

S3 server access logging records every HTTP request made to a bucket. CloudTrail and S3 access logs serve different purposes, and both are useful.

### Enable server access logging

1. Open the bucket in the S3 console
2. Click the **Properties** tab
3. Scroll to **Server access logging** and click **Edit**
4. Enable logging and choose a target bucket (use a separate bucket for logs to keep your data bucket clean)
5. Click **Save changes**

<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D02-s3-server-access-logging.png"
  alt="Server access logging settings with target bucket configured"
/>

<Note type="note" title="CloudTrail vs S3 access logs">

CloudTrail records AWS API calls at the account level, like who called `CreateBucket` or `PutBucketPolicy`. S3 server access logs record HTTP requests to a specific bucket, like who downloaded an object. Use both when you need a full audit trail.

</Note>

### VPC and S3 access

When a workload running inside a VPC needs to reach S3, traffic goes over the public internet by default. An S3 gateway endpoint changes that: it routes requests through AWS's private network instead, without leaving the AWS backbone.

The endpoint attaches to a route table in your VPC. Any instance in a subnet that uses that route table will automatically send S3 requests through the endpoint rather than the internet gateway.

Security Groups are stateful and apply at the instance level. NACLs are stateless and apply at the subnet level. Both sit between your EC2 instance and anything outside it, including S3 if you are not using a gateway endpoint.

<Note type="note">

Full VPC configuration is not required here. Knowing what a gateway endpoint does and how Security Groups differ from NACLs is enough for architecture diagrams and exam scenarios.

</Note>

### Other storage options

| Service | Stores | Typical use case | Compared to S3 |
|---|---|---|---|
| EBS | block storage | disk for one EC2 instance | behaves like a hard drive |
| EFS | file storage | shared filesystem for many EC2 instances | mounted like a network drive |
| FSx | managed filesystems | Windows, Lustre workloads | specialised file storage |
| Glacier | archive | long-term archival | slower retrieval, cheaper storage |
| Snow Family | physical transfer | move huge datasets in or out of AWS | for offline data transfer |

## Delete Buckets

If you are done with the lab, delete the bucket to avoid keeping paid resources around.


Deleting a bucket is permanent. AWS does not let you delete a bucket that still contains objects, so you need to empty it first.

### Step 1 Empty the bucket

1. Open the bucket in the S3 console
2. Click **Empty** in the bucket header, then type `permanently delete` to confirm

<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D02-s3-empty-bucket.png"
  alt="Empty bucket confirmation dialog"
/>

### Step 2 Delete the bucket

1. Go back to the S3 console home page (the bucket list)
2. Select the checkbox next to your bucket
3. Click **Delete**, then type the bucket name to confirm

<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D02-s3-delete-bucket.png"
  alt="Delete bucket confirmation dialog with bucket name input field"
/>

<Note type="warning" title="Versioned buckets need extra steps">

If versioning is enabled, deleting all objects only creates delete markers. You must also delete all versions and markers before S3 allows you to delete the bucket. The console **Empty** button handles this in one step.

</Note>

## Resources 📚📚

- [AWS S3 User Guide](https://docs.aws.amazon.com/AmazonS3/latest/userguide/)
- [S3 Storage Classes](https://aws.amazon.com/s3/storage-classes/)
- [Access Policy Language Overview](https://docs.aws.amazon.com/AmazonS3/latest/dev/access-policy-language-overview.html)
- [CORS on Amazon S3](https://docs.aws.amazon.com/AmazonS3/latest/dev/cors.html)
- [AWS VPC Overview](https://aws.amazon.com/vpc/)